In [17]:
from pathlib import Path
from typing import List
import pandas as pd
import sys
import yaml
PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))
from data_classes import roi,neuron
from data_classes.video import VideoStatistics, VideoStatisticsWriter, Video
from utils.io_utils import load_model, load_config, save_node_level_comparisons
from pipeline.io_handlers import save_filtered_suite2p, visualize_neuron_groups
from utils.visualization import print_tree
from pipeline.video_runner import VideoPipelineRunner
from experiments.tree import ExperimentTreeBuilder, is_video_dir
from experiments.processor import ExperimentProcessor
from experiments.compare import ExperimentComparer, BasicSiblingComparator


In [18]:
config_path = PROJECT_ROOT / "config" / "notebook_config.yaml"
config = load_config(config_path)
roi_model, roi_cfg = load_model(config["models"], which="roi")
spike_model, spike_cfg = load_model(config["models"], which="spike")


In [19]:

models = {
    "roi": roi_model,
    "roi_config": roi_cfg,
    "spike": spike_model,
    "spike_config": spike_cfg,
}
runner = VideoPipelineRunner.build(config)

In [20]:
builder = ExperimentTreeBuilder(is_video_dir=is_video_dir)
tree = builder.build(Path(r"C:\Users\mzinn1\Desktop\Morgan 1-20-26"))
print_tree(tree)    

└── Morgan 1-20-26
    ├── ch0
    │   └── models
    ├── ch1
    │   └── models
    ├── GCaMP6s_EX344
    │   └── week 1
    │       ├── 1-1
    │       ├── 1-2
    │       ├── 1-3
    │       ├── 1-4
    │       ├── 1-5
    │       ├── 1-6
    │       └── metrics
    ├── GCaMP6s_EX344_DL-AP5
    │   └── Week 2
    │       ├── 2-10
    │       ├── 2-10_5uM_2m
    │       ├── 2-11
    │       ├── 2-11_5uM_5m
    │       ├── 2-12
    │       ├── 2-12_5uM_8m
    │       ├── 2-13
    │       ├── 2-13_25uM_2m
    │       ├── 2-14
    │       ├── 2-14_25uM_5m
    │       ├── 2-15
    │       ├── 2-15_25uM_8m
    │       ├── 2-16
    │       ├── 2-16_50uM_2m
    │       ├── 2-17
    │       ├── 2-17_50uM_4m
    │       ├── 2-18
    │       ├── 2-18_50uM_6m
    │       ├── 2-22
    │       ├── 2-22_100uM_1m
    │       ├── 2-23
    │       ├── 2-23_100uM_4m
    │       ├── 2-24
    │       ├── 2-24_100uM_7m
    │       └── metrics
    ├── GCaMP6s_EX344_NBQX
    │   └── Week 2
    │       ├── 

In [21]:
experiment_root = r"C:\Users\mzinn1\Desktop\Morgan 1-20-26"
processor = ExperimentProcessor(runner=runner, models=models, config=config, output_root=experiment_root)
processor.process_tree(tree, verbose=True)


 Processing: 1-1
  Traces: 208 ROIs, 900 frames @ 30.0 Hz
  ROI filter: 160/208 kept (76.9%)
  Spikes: 598/3890 kept | neurons 160 → 151
  Grouping (corr): 4 groups | agreement=0.00

 Processing: 1-2
  Traces: 279 ROIs, 900 frames @ 30.0 Hz
  ROI filter: 183/279 kept (65.6%)
  Spikes: 663/4252 kept | neurons 183 → 178
  Grouping (corr): 13 groups | agreement=0.00

 Processing: 1-3
  Traces: 231 ROIs, 900 frames @ 30.0 Hz
  ROI filter: 166/231 kept (71.9%)
  Spikes: 582/3871 kept | neurons 166 → 159
  Grouping (corr): 12 groups | agreement=0.00

 Processing: 1-4
  Traces: 239 ROIs, 900 frames @ 30.0 Hz
  ROI filter: 180/239 kept (75.3%)
  Spikes: 606/4187 kept | neurons 180 → 174
  Grouping (corr): 7 groups | agreement=0.00

 Processing: 1-5
  Traces: 334 ROIs, 900 frames @ 30.0 Hz
  ROI filter: 295/334 kept (88.3%)
  Spikes: 1073/6671 kept | neurons 295 → 289
  Grouping (corr): 14 groups | agreement=0.00

 Processing: 1-6
  Traces: 211 ROIs, 900 frames @ 30.0 Hz
  ROI filter: 164/211 

In [22]:
comparer = ExperimentComparer(comparator=BasicSiblingComparator())
sibling_tables = comparer.compare_all(tree)
from experiments.io import save_node_level_comparisons_with_legend
save_node_level_comparisons_with_legend(
    root=tree,
    sibling_tables=sibling_tables,
    output_subdir="metrics",
    filename="sibling_comparisons.xlsx",
)
print("\n=== Sibling comparisons (by node) ===")
# Print the top-level node comparison if present
if experiment_root in sibling_tables:
    print(f"\nNode: {experiment_root}")
    print(sibling_tables[experiment_root].to_string(index=False))

# Print one level down comparisons too (often treatments)
for node_path, df in sibling_tables.items():
    if node_path == experiment_root:
        continue
    # keep output readable: only print “interesting” nodes
    if len(df) >= 2:
        print(f"\nNode: {node_path}")
        print(df.to_string(index=False))


=== Sibling comparisons (by node) ===

Node: C:\Users\mzinn1\Desktop\Morgan 1-20-26
                                parent                child  n_videos  n_neurons  n_groups  decay_tau_mean_unweighted  decay_tau_var_unweighted  decay_tau_within_unweighted  decay_tau_between_unweighted  decay_tau_mean_weighted  decay_tau_var_weighted  decay_tau_within_weighted  decay_tau_between_weighted  half_max_width_mean_unweighted  half_max_width_var_unweighted  half_max_width_within_unweighted  half_max_width_between_unweighted  half_max_width_mean_weighted  half_max_width_var_weighted  half_max_width_within_weighted  half_max_width_between_weighted  rise_slope_mean_unweighted  rise_slope_var_unweighted  rise_slope_within_unweighted  rise_slope_between_unweighted  rise_slope_mean_weighted  rise_slope_var_weighted  rise_slope_within_weighted  rise_slope_between_weighted  spike_frequency_mean_unweighted  spike_frequency_var_unweighted  spike_frequency_within_unweighted  spike_frequency_between_unw